In [1]:
import pandas as pd
from pathlib import Path

In [2]:
import pandas as pd
from pathlib import Path

def main():
    ROOT = Path.cwd().parents[1]
    DATA_PROCESSED = ROOT / "data_processed"

    in_path = DATA_PROCESSED / "04_final_filtered_dataset.parquet"
    out_path = DATA_PROCESSED / "05_final_filtered_adv_matches.parquet"

    print("Reading:", in_path)
    df = pd.read_parquet(in_path)

    n_rows_before = len(df)
    n_matches_before = df["match_id"].nunique()

    set_max_games = (
        df.groupby(["match_id", "SetNo"])[["P1GamesWon", "P2GamesWon"]]
          .max()
          .max(axis=1)
    )

    bad_match_ids = (
        set_max_games[set_max_games > 7]
        .index.get_level_values("match_id")
        .unique()
    )

    print("Advantage-set matches to remove:", len(bad_match_ids))

    if len(bad_match_ids) > 0:
        example = bad_match_ids[:10]
        print("Example match_ids:", list(example))

    df_clean = df[~df["match_id"].isin(bad_match_ids)].copy()

    n_rows_after = len(df_clean)
    n_matches_after = df_clean["match_id"].nunique()

    print("\nBefore:", n_rows_before, "rows |", n_matches_before, "matches")
    print("After: ", n_rows_after, "rows |", n_matches_after, "matches")
    print("Removed rows:", n_rows_before - n_rows_after)
    print("Removed matches:", n_matches_before - n_matches_after)

    max_after = (
        df_clean.groupby(["match_id", "SetNo"])[["P1GamesWon", "P2GamesWon"]]
               .max()
               .max(axis=1)
               .max()
    )
    print("\nMax games in any set after cleaning:", max_after)
    assert max_after <= 7, "Still found a set with >7 games after cleaning."

    out_path.parent.mkdir(parents=True, exist_ok=True)
    print("\nSaving:", out_path)
    df_clean.to_parquet(out_path, index=False)

    print("Done.")

In [3]:
if __name__ == "__main__":
    main()

Reading: /Users/leventezsiga/Documents/Documents - Levente’s MacBook Air/VU/Thesis_P2-P3/tennis-pressure/data_processed/04_final_filtered_dataset.parquet
Advantage-set matches to remove: 1
Example match_ids: ['2013-wimbledon-2601']

Before: 39804 rows | 278 matches
After:  39578 rows | 277 matches
Removed rows: 226
Removed matches: 1

Max games in any set after cleaning: 7

Saving: /Users/leventezsiga/Documents/Documents - Levente’s MacBook Air/VU/Thesis_P2-P3/tennis-pressure/data_processed/05_final_filtered_adv_matches.parquet
Done.
